# Analisi statistica sulla popolazione dei runner

Questo notebook documenta e lancia il job Spark per l'analisi della popolazione dei runner.

Definisco le variabili necessarie (nella reale esecuzione queste variabili vengono importate da un file di configurazione esterno).

In [ ]:
import os

CLICKHOUSE_URL = os.getenv("CLICKHOUSE_JDBC_URL", "jdbc:clickhouse://clickhouse:8123/bigintensive")
CLICKHOUSE_PROPS = {
    "user": os.getenv("CLICKHOUSE_USER", "default"),
    "password": os.getenv("CLICKHOUSE_PASSWORD", ""),
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
}
CLICKHOUSE_TABLE = os.getenv("CLICKHOUSE_TABLE", "running_samples")


POSTGRES_URL = os.getenv("POSTGRES_JDBC_URL", "jdbc:postgresql://postgres:5432/bigintensive")
POSTGRES_PROPS = {
    "user": os.getenv("POSTGRES_USER", "postgres"),
    "password": os.getenv("POSTGRES_PASSWORD", "postgres"),
    "driver": "org.postgresql.Driver",
}
POSTGRES_TABLE = os.getenv("POSTGRES_TABLE", "anthropometric_values")


KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka:19092")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC", "heart-rate-events")
KAFKA_STARTING_OFFSETS = os.getenv("SPARK_STREAM_STARTING_OFFSETS", "latest")

Importo tutte le librerie necessarie e configuro l'ambiente Spark per eseguire l'analisi dei dati.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg,  lag, countDistinct, first, row_number, to_date, radians, sin, cos, sqrt, atan2
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

Creazione di una sessione Spark con le configurazioni appropriate per l'analisi dei dati dei runner.

In [ ]:
spark = (
    SparkSession.builder.appName("running-population-analysis")
    .config("spark.master", "k8s://https://kubernetes.default:443")
    .config("spark.kubernetes.namespace", "bigintensive")
    .config("spark.kubernetes.authenticate.driver.mounted", "true")
    .config("spark.kubernetes.authenticate.driver.serviceAccountName", "spark")
    .config("spark.kubernetes.authenticate.executor.mounted", "true")
    .config("spark.driver.host", "jupyter.bigintensive.svc.cluster.local")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "7078")
    .config("spark.driver.blockManager.port", "7079")
    .config("spark.kubernetes.container.image", "apache/spark:3.5.3")
    .config("spark.executor.cores", "2")
    .config("spark.executor.memory", "2g")
    .config("spark.kubernetes.executor.request.cores", "1")
    .config("spark.kubernetes.executor.limit.cores", "2")
    .config("spark.dynamicAllocation.enabled", "true")
    .config("spark.dynamicAllocation.shuffleTracking.enabled", "true")
    .config("spark.dynamicAllocation.initialExecutors", "1")
    .config("spark.dynamicAllocation.minExecutors", "1")
    .config("spark.dynamicAllocation.maxExecutors", "4")
    .config("spark.dynamicAllocation.executorIdleTimeout", "60s")
    .config("spark.dynamicAllocation.cachedExecutorIdleTimeout", "120s")
    # clickhouse-jdbc non e' shaded: senza http-client e httpclient5 il driver non si carica.
    .config(
        "spark.jars.packages",
        "org.postgresql:postgresql:42.7.3,"
        "com.clickhouse:clickhouse-jdbc:0.6.3,"
        "com.clickhouse:clickhouse-http-client:0.6.3,"
        "org.apache.httpcomponents.client5:httpclient5:5.3.1",
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Importiamo le tabelle dei dati dei runner da clickhouse e Postgresql.

In [ ]:
df = (
    spark.read.format("jdbc")
    .option("url", CLICKHOUSE_URL)
    .option("dbtable", CLICKHOUSE_TABLE)
    .option("user", CLICKHOUSE_PROPS["user"])
    .option("password", CLICKHOUSE_PROPS["password"])
    .option("driver", CLICKHOUSE_PROPS["driver"])
    .load()
)

df_postgres = (
    spark.read.format("jdbc")
    .option("url", POSTGRES_URL)
    .option("dbtable", POSTGRES_TABLE)
    .option("user", POSTGRES_PROPS["user"])
    .option("password", POSTGRES_PROPS["password"])
    .option("driver", POSTGRES_PROPS["driver"])
    .load()
)

df.show(5)
df_postgres.show(5)

Creazione delle finestre temporali per l'analisi dei dati dei runner, utilizzando le funzioni di finestra di Spark.

In [ ]:
finestra_temporale = Window.partitionBy("athlete_id", "session_id").orderBy("sample_id")
finestra_temporale_5min = finestra_temporale.rowsBetween(-60, 0)

Pulisco il dataset da valori nulli e duplicati.

In [ ]:
df_ordinato = df.orderBy(col("athlete_id"), col("session_id"), col("sample_id"))
df_ordinato.show(5)

df_pulito = df_ordinato.dropDuplicates(["athlete_id", "session_id", "sample_id"])
df_pulito.show(5)

df_pulito_null = df_pulito.filter(
    col("athlete_id").isNotNull()
    & col("session_id").isNotNull()
    & col("sample_id").isNotNull()
    & col("heart_rate").isNotNull()
    & col("latitude").isNotNull()
    & col("longitude").isNotNull()
    & col("timestamp").isNotNull()
)
df_pulito_null.show(5)

Aggiungo e calcolo la colonna BMI (Body Mass Index) nel dataset dei runner, utilizzando i dati di peso e altezza.

In [ ]:
df_postgres_aggiornato = df_postgres.withColumn("BMI", col("peso_kg") / (col("altezza_cm") * col("altezza_cm")/10000))
df_postgres_aggiornato.show(5)

Pulisco il dataset di Postgres selezionando solo le colonne necessarie per l'analisi, dopo aver calcolato il BMI.

In [ ]:
df_postgres_ridotto = df_postgres_aggiornato.select("athlete_id", "data_rilevazione", "peso_kg", "altezza_cm", "BMI")
df_postgres_ridotto.show(5)

Calcolo della velocità media tramite la posizione GPS tramite le formule trigonometriche distribuite e della deriva cardiaca puntuale e percentuale per ogni atleta e sessione, usando le finestre appena create.

In [ ]:
R=6371000  # Raggio della Terra in metri
df_deriva_cardiaca = df_pulito_null \
    .withColumn("lat_prec", lag("latitude", 1).over(finestra_temporale)) \
    .withColumn("lon_prec", lag("longitude", 1).over(finestra_temporale)) \
    .withColumn("lat_rad", radians(col("latitude"))) \
    .withColumn("lon_rad", radians(col("longitude"))) \
    .withColumn("lat_prec_rad", radians(col("lat_prec"))) \
    .withColumn("lon_prec_rad", radians(col("lon_prec"))) \
    .withColumn("dlat", col("lat_rad") - col("lat_prec_rad")) \
    .withColumn("dlon", col("lon_rad") - col("lon_prec_rad")) \
    .withColumn("a", sin(col("dlat") / 2) ** 2 + cos(col("lat_rad")) * cos(col("lat_prec_rad")) * sin(col("dlon") / 2) ** 2) \
    .withColumn("c", 2 * atan2(sqrt(col("a")), sqrt(1 - col("a")))) \
    .withColumn("distanza", R * col("c")) \
    .withColumn("velocita_puntuale", col("distanza") / 5) \
    .withColumn("velocita_media", avg("velocita_puntuale").over(finestra_temporale_5min)) \
    .withColumn("frequenza_cardiaca_media", avg("heart_rate").over(finestra_temporale_5min)) \
    .withColumn("Efficienza_puntuale", col("velocita_puntuale") / col("frequenza_cardiaca_media")) \
    .withColumn("Efficienza_puntuale_iniziale", first("Efficienza_puntuale").over(finestra_temporale)) \
    .withColumn("Deriva_cardiaca_percentuale", (col("Efficienza_puntuale")- col("Efficienza_puntuale_iniziale")) / col("Efficienza_puntuale_iniziale") * 100)


Trovo la velocità che causa la deriva cardiaca più alta per ogni atleta e sessione, filtrando i dati per valori di deriva cardiaca superiori al 5%. Ne estrae il primo punto di crisi per ogni sessione.

In [ ]:
finestra_sessione_tempo = Window.partitionBy("athlete_id", "session_id").orderBy("timestamp")
df_crisi_ordinate = df_deriva_cardiaca \
                    .filter(col("Deriva_cardiaca_percentuale") > 5) \
                    .withColumn("riga_crisi", row_number().over(finestra_sessione_tempo))
                    
primo_punto_di_crisi_per_sessione = df_crisi_ordinate \
        .filter(col("riga_crisi") == 1) \
        .select("timestamp", "velocita_media", "Deriva_cardiaca_percentuale", "athlete_id", "session_id")

Calcolo il numero di corse per ogni atleta e unisco questo dato al dataset dei punti di crisi.

In [ ]:
df_conteggio_corse = primo_punto_di_crisi_per_sessione.groupBy("athlete_id").agg(countDistinct("session_id").alias("numero_corse"))

Eseguo il join con il dataset di Postgres e delle corse per ottenere un dataset finale con tutte le informazioni necessarie per l'analisi della popolazione dei runner.

In [ ]:
primo_punto_di_crisi_per_sessione = primo_punto_di_crisi_per_sessione.join(
        df_postgres_ridotto,
        (primo_punto_di_crisi_per_sessione["athlete_id"] == df_postgres_ridotto["athlete_id"]) & (to_date(col("timestamp")) == col("data_rilevazione")),
        how="inner"
).drop(df_postgres_ridotto["athlete_id"])

df_preanalisi = primo_punto_di_crisi_per_sessione.join(df_conteggio_corse, ["athlete_id"], how="left")

Calcolo la matrice di correlazione tra le variabili del dataset finale per identificare eventuali relazioni tra le caratteristiche dei runner e i punti di crisi.

In [ ]:
colonne_da_analizzare = ["peso_kg", "altezza_cm", "BMI", "velocita_media","numero_corse"]

df_ml = df_preanalisi.select(colonne_da_analizzare).na.drop()

assembler = VectorAssembler(inputCols=colonne_da_analizzare, outputCol="features")
df_ml = assembler.transform(df_ml)

matrice_correlazione = Correlation.corr(df_ml, "features","pearson").head()[0]

print("Matrice di correlazione:")
print(matrice_correlazione)

## Chiudi la sessione Spark

Esegui questa cella solo quando hai concluso tutte le analisi nel notebook.

In [ ]:
spark.stop()
print("SparkSession closed.")